In [ ]:
import numpy as np

In [ ]:
Q = np.array([
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 1.0, 0.0, 1.0],
    [1.0, 1.0, 0.0, 0.0]
])

K = np.array([
    [1.0, 1.0, 0.0, 0.0],
    [0.0, 1.0, 1.0, 0.0],
    [1.0, 0.0, 0.0, 1.0]
])

V = np.array([
    [10.0, 0.0, 0.0, 10.0],
    [0.0, 20.0, 20.0, 0.0],
    [5.0, 5.0, 5.0, 5.0]
])   # Fallback for unknown words: an array of zeros
dk = 4.0
scale_factor = np.sqrt(dk)

In [ ]:
import gensim.downloader as api
word_vectors = api.load("glove-wiki-gigaword-50")

In [ ]:
class Encoder:
  def __init__(self, dim, hid_dim):
    self.dim = dim
    self.hid_dim = hid_dim

    self.w1 = np.random.randn(dim, hid_dim) * 0.1
    self.b1 = np.zeros((hid_dim,))

    self.w2 = np.random.randn(hid_dim, dim) * 0.1
    self.b2 = np.zeros((dim,))

  def softmax(self, x):
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / e_x.sum(axis=-1, keepdims=True)

  def self_attention(self, Q, K, V, scale_factor):
    scores = np.dot(Q, K.T) / scale_factor
    out = np.dot(self.softmax(scores), V)
    return out

  def feed_forward(self, x):
    z1 = np.dot(x, self.w1) + self.b1
    a1 = np.tanh(z1)
    z2 = np.dot(a1, self.w2) + self.b2
    return z2

  def layer_norm(self, x, gamma, beta, eps=1e-5):
    mean = np.mean(x, axis=-1, keepdims=True)
    std = np.std(x, axis=-1, keepdims=True)
    return gamma * (x - mean) / (std + eps) + beta

In [ ]:
class Decoder:
  def __init__(self, dim, hid_dim, embedding_dim):
    self.w1 = np.random.randn(dim, hid_dim) * 0.1
    self.b1 = np.zeros((hid_dim,))

    self.w2 = np.random.randn(hid_dim, dim) * 0.1
    self.b2 = np.zeros((dim,))

    # New projection layer to map decoder output to embedding_dim
    self.output_proj_w = np.random.randn(dim, embedding_dim) * 0.1
    self.output_proj_b = np.zeros((embedding_dim,))

  def softmax(self, x):
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / e_x.sum(axis=-1, keepdims=True)

  def self_attention(self, Q, K, V, scale_factor):
    scores = np.dot(Q, K.T) / scale_factor
    out = np.dot(self.softmax(scores), V)
    return out

  def feed_forward(self, x):
    z1 = np.dot(x, self.w1) + self.b1
    a1 = np.tanh(z1)
    z2 = np.dot(a1, self.w2) + self.b2
    return z2

  def layer_norm(self, x, gamma, beta, eps=1e-5):
    mean = np.mean(x, axis=-1, keepdims=True)
    std = np.std(x, axis=-1, keepdims=True)
    return gamma * (x - mean) / (std + eps) + beta

  def cross_attention(self, Q, K, V, scale_factor):
    scores = np.dot(Q, K.T) / scale_factor
    output = np.dot(self.softmax(scores), V)
    return output

In [ ]:
user_text = input("Enter a string")
tokens = user_text.lower().split()
embedding_dim = 50
embedded_sequence = []

for token in tokens:
    if token in word_vectors:
        embedded_sequence.append(word_vectors[token])
    else:
        embedded_sequence.append(np.zeros(embedding_dim))
embedded_sequence = np.array(embedded_sequence)

Enter a stringthis is a test string


In [ ]:
encoder = Encoder(dim=4, hid_dim=50)
decoder = Decoder(dim=4, hid_dim=50, embedding_dim=embedding_dim)
final_weight =embedded_sequence.T

In [ ]:
for _ in range(5):
    x = encoder.self_attention(Q, K, V, scale_factor)
    ln1 = encoder.layer_norm(x, gamma=np.ones(4), beta=np.zeros(4))
    x = ln1 + x
    encoder_output = encoder.feed_forward(x)
    ln2 = encoder.layer_norm(encoder_output, gamma=np.ones(4), beta=np.zeros(4))
    x = ln2 + encoder_output
    dec_attn = decoder.self_attention(Q, K, V, scale_factor)
    ln3 = decoder.layer_norm(dec_attn, gamma=np.ones(4), beta=np.zeros(4))
    dec_attn = ln3 + dec_attn
    cross_out = decoder.cross_attention(dec_attn, encoder_output, encoder_output, scale_factor)
    ln4 = decoder.layer_norm(cross_out, gamma=np.ones(4), beta=np.zeros(4))
    cross_out = ln4 + cross_out
    decoder_output = decoder.feed_forward(cross_out)
    ln5 = decoder.layer_norm(decoder_output, gamma=np.ones(4), beta=np.zeros(4))
    decoder_output = ln5 + decoder_output

    # Project decoder_output from model_dim (4) to embedding_dim (50)
    projected_decoder_output = np.dot(decoder_output, decoder.output_proj_w) + decoder.output_proj_b

    # Now, multiply projected_decoder_output (3, 50) with final_weight (50, 4)
    linear = np.dot(projected_decoder_output, final_weight)
    final_output = encoder.softmax(linear)
print(final_output)

[[0.1453126  0.23221105 0.14419201 0.26756906 0.21071528]
 [0.1453126  0.23221105 0.14419201 0.26756906 0.21071528]
 [0.1453126  0.23221105 0.14419201 0.26756906 0.21071528]]
